In [0]:
CREATE TABLE IF NOT EXISTS ecom_dev.gold.dim_date (
  date_key      INT     NOT NULL,
  full_date     DATE    NOT NULL,
  year          INT,
  quarter       INT,
  month         INT,
  month_name    STRING,
  year_month    STRING,
  week_of_year  INT,
  day_of_month  INT,
  day_of_week   INT,
  day_name      STRING,
  is_weekend    BOOLEAN
) COMMENT 'One row per calendar day (Monday = 1 ... Sunday = 7)';

INSERT OVERWRITE ecom_dev.gold.dim_date
SELECT CAST(date_format(d, 'yyyyMMdd') AS INT) AS date_key,
       d                                       AS full_date,
       year(d), quarter(d), month(d),
       date_format(d, 'MMMM')                  AS month_name,
       date_format(d, 'yyyy-MM')               AS year_month,
       weekofyear(d), day(d),
       ((dayofweek(d) + 5) % 7) + 1            AS day_of_week,
       date_format(d, 'EEEE')                  AS day_name,
       dayofweek(d) IN (1, 7)                  AS is_weekend
FROM (SELECT explode(sequence(to_date('2024-01-01'), to_date('2028-12-31'), interval 1 day)) AS d);

SELECT count(*) AS days, min(full_date) AS first_day, max(full_date) AS last_day FROM ecom_dev.gold.dim_date;
SELECT * FROM ecom_dev.gold.dim_date WHERE full_date = '2025-12-25';

CREATE TABLE IF NOT EXISTS ecom_dev.gold.dim_seller (
  seller_key BIGINT NOT NULL, seller_id STRING NOT NULL, seller_name STRING,
  city STRING, state STRING, seller_since DATE, seller_rating DECIMAL(3,1)
);

INSERT OVERWRITE ecom_dev.gold.dim_seller
SELECT xxhash64(seller_id), seller_id, seller_name, city, state, seller_since, seller_rating
FROM ecom_dev.silver.sellers
UNION ALL
SELECT -1, 'UNKNOWN', 'Unknown', NULL, NULL, NULL, NULL;

CREATE TABLE IF NOT EXISTS ecom_dev.gold.dim_product (
  product_key BIGINT NOT NULL, product_id STRING NOT NULL, product_name STRING, category STRING,
  brand STRING, unit_price DECIMAL(12,2), cost_price DECIMAL(12,2), is_active BOOLEAN,
  seller_id STRING, valid_from TIMESTAMP NOT NULL, valid_to TIMESTAMP NOT NULL, is_current BOOLEAN
);

INSERT OVERWRITE ecom_dev.gold.dim_product
SELECT xxhash64(product_id, CAST(valid_from AS STRING)), product_id, product_name, category, brand,
       unit_price, cost_price, is_active, seller_id, valid_from, valid_to,
       valid_to = TIMESTAMP'9999-12-31 00:00:00'
FROM (
  SELECT *,
         CASE WHEN row_number() OVER (PARTITION BY product_id ORDER BY updated_at) = 1
              THEN TIMESTAMP'1900-01-01 00:00:00' ELSE updated_at END AS valid_from,
         coalesce(lead(updated_at) OVER (PARTITION BY product_id ORDER BY updated_at),
                  TIMESTAMP'9999-12-31 00:00:00') AS valid_to
  FROM ecom_dev.silver.products
)
UNION ALL
SELECT -1, 'UNKNOWN', 'Unknown', NULL, NULL, NULL, NULL, NULL, NULL,
       TIMESTAMP'1900-01-01 00:00:00', TIMESTAMP'9999-12-31 00:00:00', true;

CREATE TABLE IF NOT EXISTS ecom_dev.gold.dim_customer (
  customer_key BIGINT NOT NULL, customer_id STRING NOT NULL, customer_name STRING, email STRING,
  phone STRING, city STRING, state STRING, signup_date DATE,
  valid_from TIMESTAMP NOT NULL, valid_to TIMESTAMP NOT NULL, is_current BOOLEAN
);

INSERT OVERWRITE ecom_dev.gold.dim_customer
SELECT xxhash64(customer_id, CAST(valid_from AS STRING)), customer_id, customer_name, email, phone,
       city, state, signup_date, valid_from, valid_to,
       valid_to = TIMESTAMP'9999-12-31 00:00:00'
FROM (
  SELECT *,
         CASE WHEN row_number() OVER (PARTITION BY customer_id ORDER BY updated_at) = 1
              THEN TIMESTAMP'1900-01-01 00:00:00' ELSE updated_at END AS valid_from,
         coalesce(lead(updated_at) OVER (PARTITION BY customer_id ORDER BY updated_at),
                  TIMESTAMP'9999-12-31 00:00:00') AS valid_to
  FROM ecom_dev.silver.customers
)
UNION ALL
SELECT -1, 'UNKNOWN', 'Unknown', NULL, NULL, NULL, NULL, NULL,
       TIMESTAMP'1900-01-01 00:00:00', TIMESTAMP'9999-12-31 00:00:00', true;


CREATE TABLE IF NOT EXISTS ecom_dev.gold.fact_order_items (
  order_id STRING NOT NULL, order_item_id INT NOT NULL, date_key INT, order_ts TIMESTAMP,
  customer_key BIGINT, product_key BIGINT, seller_key BIGINT, order_status STRING, channel STRING,
  quantity INT, unit_price DECIMAL(12,2), discount_amount DECIMAL(12,2), freight_value DECIMAL(12,2),
  line_amount DECIMAL(14,2), updated_at TIMESTAMP, _loaded_ts TIMESTAMP
);

MERGE INTO ecom_dev.gold.fact_order_items t
USING (
  SELECT i.order_id, i.order_item_id,
         CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS date_key,
         o.order_purchase_ts AS order_ts,
         coalesce(c.customer_key, -1) AS customer_key,
         coalesce(p.product_key, -1)  AS product_key,
         coalesce(s.seller_key, -1)   AS seller_key,
         o.order_status, o.channel,
         i.quantity, i.unit_price, i.discount_amount, i.freight_value, i.line_amount,
         i.updated_at, current_timestamp() AS _loaded_ts
  FROM ecom_dev.silver.order_items i
  JOIN ecom_dev.silver.orders o ON o.order_id = i.order_id
  LEFT JOIN ecom_dev.gold.dim_customer c ON c.customer_id = o.customer_id
       AND o.order_purchase_ts >= c.valid_from AND o.order_purchase_ts < c.valid_to
  LEFT JOIN ecom_dev.gold.dim_product p ON p.product_id = i.product_id
       AND o.order_purchase_ts >= p.valid_from AND o.order_purchase_ts < p.valid_to
  LEFT JOIN ecom_dev.gold.dim_seller s ON s.seller_id = i.seller_id
) s
ON t.order_id = s.order_id AND t.order_item_id = s.order_item_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

CREATE TABLE IF NOT EXISTS ecom_dev.gold.fact_orders (
  order_id STRING NOT NULL, date_key INT, customer_key BIGINT, order_status STRING, channel STRING,
  purchase_ts TIMESTAMP, approved_ts TIMESTAMP, shipped_ts TIMESTAMP, delivered_ts TIMESTAMP,
  estimated_delivery_date DATE, items_count BIGINT, items_amount DECIMAL(18,2),
  freight_amount DECIMAL(18,2), paid_amount DECIMAL(18,2), delivery_days INT,
  is_delivered BOOLEAN, is_late BOOLEAN, updated_at TIMESTAMP, _loaded_ts TIMESTAMP
);

MERGE INTO ecom_dev.gold.fact_orders t
USING (
  SELECT o.order_id,
         CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS date_key,
         coalesce(c.customer_key, -1) AS customer_key,
         o.order_status, o.channel,
         o.order_purchase_ts AS purchase_ts, o.order_approved_ts AS approved_ts,
         o.order_shipped_ts AS shipped_ts, o.order_delivered_ts AS delivered_ts,
         o.estimated_delivery_date,
         CAST(coalesce(i.items_count, 0) AS BIGINT)        AS items_count,
         CAST(coalesce(i.items_amount, 0) AS DECIMAL(18,2)) AS items_amount,
         CAST(coalesce(i.freight_amount, 0) AS DECIMAL(18,2)) AS freight_amount,
         CAST(coalesce(p.paid_amount, 0) AS DECIMAL(18,2))  AS paid_amount,
         datediff(o.order_delivered_ts, o.order_purchase_ts) AS delivery_days,
         o.order_delivered_ts IS NOT NULL AS is_delivered,
         coalesce(to_date(o.order_delivered_ts) > o.estimated_delivery_date, false) AS is_late,
         o.updated_at, current_timestamp() AS _loaded_ts
  FROM ecom_dev.silver.orders o
  LEFT JOIN ecom_dev.gold.dim_customer c ON c.customer_id = o.customer_id
       AND o.order_purchase_ts >= c.valid_from AND o.order_purchase_ts < c.valid_to
  LEFT JOIN (SELECT order_id, sum(quantity) AS items_count, sum(line_amount) AS items_amount,
                    sum(freight_value) AS freight_amount
             FROM ecom_dev.silver.order_items GROUP BY order_id) i ON i.order_id = o.order_id
  LEFT JOIN (SELECT order_id, sum(payment_value) AS paid_amount
             FROM ecom_dev.silver.payments GROUP BY order_id) p ON p.order_id = o.order_id
) s
ON t.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

CREATE TABLE IF NOT EXISTS ecom_dev.gold.fact_payments (
  order_id STRING NOT NULL, payment_sequential INT NOT NULL, date_key INT, payment_type STRING,
  payment_installments INT, payment_value DECIMAL(12,2), updated_at TIMESTAMP, _loaded_ts TIMESTAMP
);

MERGE INTO ecom_dev.gold.fact_payments t
USING (
  SELECT p.order_id, p.payment_sequential,
         CAST(date_format(o.order_date, 'yyyyMMdd') AS INT) AS date_key,
         p.payment_type, p.payment_installments, p.payment_value, p.updated_at,
         current_timestamp() AS _loaded_ts
  FROM ecom_dev.silver.payments p
  LEFT JOIN ecom_dev.silver.orders o ON o.order_id = p.order_id
) s
ON t.order_id = s.order_id AND t.payment_sequential = s.payment_sequential
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

CREATE TABLE IF NOT EXISTS ecom_dev.gold.fact_reviews (
  review_id STRING NOT NULL, order_id STRING, date_key INT, review_score INT,
  has_comment BOOLEAN, review_comment STRING, review_ts TIMESTAMP, _loaded_ts TIMESTAMP
);

MERGE INTO ecom_dev.gold.fact_reviews t
USING (
  SELECT review_id, order_id,
         CAST(date_format(review_ts, 'yyyyMMdd') AS INT) AS date_key,
         review_score, review_comment IS NOT NULL AS has_comment, review_comment, review_ts,
         current_timestamp() AS _loaded_ts
  FROM ecom_dev.silver.reviews
) s
ON t.review_id = s.review_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


SELECT 'order_items: gold vs silver' AS check_name,
       (SELECT count(*) FROM ecom_dev.gold.fact_order_items) AS gold_value,
       (SELECT count(*) FROM ecom_dev.silver.order_items)    AS expected
UNION ALL SELECT 'order_items with no order (left out)',
       (SELECT count(*) FROM ecom_dev.silver.order_items i LEFT ANTI JOIN ecom_dev.silver.orders o ON o.order_id = i.order_id), 0
UNION ALL SELECT 'orders: gold vs silver',
       (SELECT count(*) FROM ecom_dev.gold.fact_orders), (SELECT count(*) FROM ecom_dev.silver.orders)
UNION ALL SELECT 'payments: gold vs silver',
       (SELECT count(*) FROM ecom_dev.gold.fact_payments), (SELECT count(*) FROM ecom_dev.silver.payments)
UNION ALL SELECT 'reviews: gold vs silver',
       (SELECT count(*) FROM ecom_dev.gold.fact_reviews), (SELECT count(*) FROM ecom_dev.silver.reviews)
UNION ALL SELECT 'revenue: gold items vs fact_orders (diff)',
       CAST((SELECT sum(line_amount) FROM ecom_dev.gold.fact_order_items)
          - (SELECT sum(items_amount) FROM ecom_dev.gold.fact_orders) AS BIGINT), 0
UNION ALL SELECT 'lines pointing to Unknown product',
       (SELECT count(*) FROM ecom_dev.gold.fact_order_items WHERE product_key = -1), 0
UNION ALL SELECT 'lines pointing to Unknown customer',
       (SELECT count(*) FROM ecom_dev.gold.fact_order_items WHERE customer_key = -1), 0
UNION ALL SELECT 'lines with no date_key',
       (SELECT count(*) FROM ecom_dev.gold.fact_order_items WHERE date_key IS NULL), 0
UNION ALL SELECT 'duplicate dim keys (product+customer)',
       (SELECT count(*) - count(DISTINCT product_key) FROM ecom_dev.gold.dim_product)
     + (SELECT count(*) - count(DISTINCT customer_key) FROM ecom_dev.gold.dim_customer), 0;

INSERT INTO ecom_dev.control.watermark
  (table_name, layer, watermark_column, last_watermark_value, last_run_id, last_success_ts, updated_at)
SELECT * FROM VALUES
  ('gold.orders_group', 'gold', 'silver._processed_ts', '1900-01-01 00:00:00', NULL, NULL, current_timestamp()),
  ('gold.fact_reviews', 'gold', 'silver._processed_ts', '1900-01-01 00:00:00', NULL, NULL, current_timestamp())
WHERE NOT EXISTS (SELECT 1 FROM ecom_dev.control.watermark WHERE table_name = 'gold.orders_group');